In [23]:
from manim import *
import os
import numpy as np

class MultiHeadAttentionScene(Scene):
    def construct(self):
        # ==========================================
        # 0. CONFIG & HELPERS
        # ==========================================
        self.camera.background_color = "#1e1e1e"
        
        # Helper: Clean Matrix Generation
        def get_matrix(data, label, color, scale=0.40):
            m = Matrix(data, v_buff=0.5, h_buff=0.7, bracket_h_buff=0.1).set_column_colors(color)
            m.scale(scale)
            lbl = MathTex(label, color=color).scale(scale).next_to(m, UP, buff=0.1)
            return VGroup(lbl, m)

        # Helper: Robust Highlight Line
        def get_code_lines_group(code_mobj):
            if hasattr(code_mobj, 'code'):
                return code_mobj.code
            elif len(code_mobj) >= 3:
                return code_mobj[2]
            elif len(code_mobj) >= 2:
                return code_mobj[1]
            return None

        def highlight_lines(line_indices):
            lines = get_code_lines_group(code_obj)
            if lines is None: return VGroup()
            
            grp = VGroup()
            for idx in line_indices:
                if 0 <= idx < len(lines):
                    target_line = lines[idx]
                    rect = SurroundingRectangle(target_line, color=YELLOW, stroke_width=2, buff=0.05)
                    rect.stretch(1.05, 0) 
                    grp.add(rect)
            return grp

        # ==========================================
        # 1. GENERATE CODE
        # ==========================================
        source_content = """import torch
X = torch.tensor([[1., 0.], [0., 1.]])

# 1. Heads (Learnable Weights)
# Head 1 (Blue)
Wq1 = torch.rand(2,2); Wk1 = torch.rand(2,2)
Wv1 = torch.rand(2,2)
Q1, K1, V1 = X@Wq1, X@Wk1, X@Wv1

# Head 2 (Red)
Wq2, Wk2, Wv2 = [torch.rand(2,2) for _ in range(3)]
Q2, K2, V2 = X@Wq2, X@Wk2, X@Wv2

# 2. Scaled Dot-Product
# Scale = sqrt(d_k)
S1 = (Q1 @ K1.T) / 1.41
S2 = (Q2 @ K2.T) / 1.41

# 3. Softmax & Context
A1 = torch.softmax(S1, dim=-1)
A2 = torch.softmax(S2, dim=-1)
Z1, Z2 = A1 @ V1, A2 @ V2

# 4. Concat & Output (Learnable Wo)
Z_cat = torch.cat([Z1, Z2], dim=1)
Wo = torch.rand(4, 2)
Out = Z_cat @ Wo
"""
        filename = "mha_final_v6.py"
        with open(filename, "w") as f:
            f.write(source_content)

        # ==========================================
        # 2. LAYOUT SETUP
        # ==========================================
        
        # --- TITLE (Top Left) ---
        main_title = Text("Multi-Head Attention", font_size=32, weight=BOLD, color=WHITE)
        main_title.to_corner(UL, buff=0.5)
        
        # Code section
        code_obj = Code(filename, language="python")
        code_obj.scale(0.48) 
        code_obj.next_to(main_title, DOWN, buff=0.2, aligned_edge=LEFT)
        
        if len(code_obj) > 2: 
            code_obj[1].set_opacity(0) 
        
        # Gray out comments
        code_lines = get_code_lines_group(code_obj)
        if code_lines:
            source_lines = source_content.split('\n')
            for i, line in enumerate(code_lines):
                if i < len(source_lines) and source_lines[i].strip().startswith("#"):
                    line.set_color(GRAY)

        # --- DIVIDER ---
        divider = Line(UP*3.5, DOWN*3.5, color=GRAY_D).next_to(code_obj, RIGHT, buff=0.5)
        
        # --- TOP HEADER (Formulas) ---
        formula_group = VGroup(
            MathTex(r"\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V", font_size=24),
            MathTex(r"\text{MHA}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O", font_size=24)
        ).arrange(DOWN, buff=0.2, aligned_edge=LEFT)
        
        formula_group.next_to(divider, RIGHT, buff=0.5).to_edge(UP, buff=0.5)
        
        # Input X
        x_raw = [[1, 0], [0, 1]]
        mat_X = get_matrix(x_raw, "Input X", WHITE, scale=0.5)
        mat_X.next_to(formula_group, RIGHT, buff=1.0)
        
        self.add(main_title, code_obj, divider, formula_group)

        # Explanation Text Center
        RIGHT_PANEL_CENTER = (divider.get_x() + config.frame_width/2) / 2 + 0.5
        
        expl_text = Text("Initializing...", font_size=22, color=WHITE)
        expl_text.move_to([RIGHT_PANEL_CENTER, -3.5, 0])
        self.add(expl_text)

        # ==========================================
        # 3. ANIMATION SEQUENCING
        # ==========================================
        curr_hl = VGroup()

        def update_expl(text):
            new_t = Text(text, font_size=20, color=YELLOW).move_to(expl_text)
            if new_t.width > 6: new_t.scale_to_fit_width(6)
            return Transform(expl_text, new_t)

        # --- STEP 0: INPUT ---
        hl = highlight_lines([1]) 
        self.play(Create(hl), FadeIn(mat_X), update_expl("Step 0: Input Tensor X"))
        curr_hl = hl
        self.wait(0.5)

        # --- STEP 1: WEIGHT MATRICES ---
        hl_w = highlight_lines([5, 6, 10]) 
        self.play(
            ReplacementTransform(curr_hl, hl_w),
            update_expl("Step 1: Init Learnable Weights (Wq, Wk, Wv)")
        )
        curr_hl = hl_w

        H1_X = RIGHT_PANEL_CENTER - 1.5
        H2_X = RIGHT_PANEL_CENTER + 1.5
        Y_ROW_1 = 1.2 

        wq1 = get_matrix([[1, 0], [0, 1]], "W_{q1}", BLUE_B, 0.40).move_to([H1_X - 1.0, Y_ROW_1, 0])
        wk1 = get_matrix([[1, 0], [0, 1]], "W_{k1}", BLUE_B, 0.40).move_to([H1_X, Y_ROW_1, 0])
        wv1 = get_matrix([[2, 0], [0, 2]], "W_{v1}", BLUE_B, 0.40).move_to([H1_X + 1.0, Y_ROW_1, 0])
        
        wq2 = get_matrix([[0, 1], [1, 0]], "W_{q2}", RED_B, 0.40).move_to([H2_X - 1.0, Y_ROW_1, 0])
        wk2 = get_matrix([[0, 1], [1, 0]], "W_{k2}", RED_B, 0.40).move_to([H2_X, Y_ROW_1, 0])
        wv2 = get_matrix([[3, 0], [0, 3]], "W_{v2}", RED_B, 0.40).move_to([H2_X + 1.0, Y_ROW_1, 0])

        self.play(FadeIn(wq1), FadeIn(wk1), FadeIn(wv1), FadeIn(wq2), FadeIn(wk2), FadeIn(wv2))

        # --- STEP 2: PROJECTIONS ---
        hl_proj = highlight_lines([7, 11])
        self.play(
             ReplacementTransform(curr_hl, hl_proj),
             update_expl("Step 2: Project X @ W -> Q, K, V")
        )
        curr_hl = hl_proj

        q1 = get_matrix([[1, 0], [0, 1]], "Q_1", BLUE, 0.40).move_to(wq1)
        k1 = get_matrix([[1, 0], [0, 1]], "K_1", BLUE, 0.40).move_to(wk1)
        v1 = get_matrix([[2, 0], [0, 2]], "V_1", BLUE_C, 0.40).move_to(wv1)
        
        q2 = get_matrix([[0, 1], [1, 0]], "Q_2", RED, 0.40).move_to(wq2)
        k2 = get_matrix([[0, 1], [1, 0]], "K_2", RED, 0.40).move_to(wk2)
        v2 = get_matrix([[3, 0], [0, 3]], "V_2", RED_C, 0.40).move_to(wv2)

        self.play(
            ReplacementTransform(wq1, q1), ReplacementTransform(wk1, k1), ReplacementTransform(wv1, v1),
            ReplacementTransform(wq2, q2), ReplacementTransform(wk2, k2), ReplacementTransform(wv2, v2),
        )

        # --- STEP 3: SCORES ---
        hl_score = highlight_lines([15, 16])
        self.play(
            ReplacementTransform(curr_hl, hl_score),
            update_expl("Step 3: Score = (Q @ K.T) / scale")
        )
        curr_hl = hl_score

        s1 = get_matrix([[1, 0], [0, 1]], "S_1", BLUE, 0.40).move_to(k1.get_center())
        s2 = get_matrix([[1, 0], [0, 1]], "S_2", RED, 0.40).move_to(k2.get_center())

        self.play(
            FadeOut(q1), FadeOut(q2),
            ReplacementTransform(k1, s1),
            ReplacementTransform(k2, s2),
            v1.animate.next_to(s1, RIGHT, buff=0.15),
            v2.animate.next_to(s2, RIGHT, buff=0.15)
        )

        # --- STEP 4: SOFTMAX & CONTEXT ---
        hl_soft = highlight_lines([19, 20, 21])
        self.play(
            ReplacementTransform(curr_hl, hl_soft),
            update_expl("Step 4: Softmax(Score) @ V")
        )
        curr_hl = hl_soft

        p1 = get_matrix([[0.5, 0.5], [0.5, 0.5]], "A_1", BLUE, 0.40).move_to(s1)
        p2 = get_matrix([[0.5, 0.5], [0.5, 0.5]], "A_2", RED, 0.40).move_to(s2)
        self.play(Transform(s1, p1), Transform(s2, p2))
        
        Y_ROW_2 = -0.5 
        z1 = get_matrix([[1.2, 0], [0, 1.2]], "Z_1", BLUE, 0.45).move_to([H1_X, Y_ROW_2, 0])
        z2 = get_matrix([[0, 1.8], [1.8, 0]], "Z_2", RED, 0.45).move_to([H2_X, Y_ROW_2, 0])

        self.play(
            ReplacementTransform(VGroup(s1, v1), z1),
            ReplacementTransform(VGroup(s2, v2), z2)
        )

        # --- STEP 5: CONCAT ---
        hl_cat = highlight_lines([24])
        self.play(
            ReplacementTransform(curr_hl, hl_cat),
            update_expl("Step 5: Concatenate Heads")
        )
        curr_hl = hl_cat

        concat_center = [RIGHT_PANEL_CENTER, -0.5, 0]
        self.play(
            z1.animate.move_to(np.array(concat_center) + LEFT*0.7),
            z2.animate.move_to(np.array(concat_center) + RIGHT*0.7)
        )
        
        # Concat Visuals
        box = SurroundingRectangle(VGroup(z1, z2), color=PURPLE, buff=0.15)
        # ADDED: Concat label above the box
        concat_label = Text("Concat", font_size=24, color=PURPLE).next_to(box, UP, buff=0.1)
        
        self.play(Create(box), Write(concat_label))

        # --- STEP 6: OUTPUT ---
        hl_out = highlight_lines([25, 26])
        self.play(
            ReplacementTransform(curr_hl, hl_out),
            update_expl("Step 6: Output Project (Learnable Wo)")
        )

        wo_math = MathTex(r"\times W_o", color=PURPLE).next_to(box, RIGHT)
        self.play(Write(wo_math))

        final_out = get_matrix([[1, 0], [0, 1]], "Output", GREEN, 0.5)
        final_out.next_to(box, DOWN, buff=0.8) 
        
        # Check collision with description text
        # visual_group includes the new concat_label
        if final_out.get_bottom()[1] < expl_text.get_top()[1]:
             visual_group = VGroup(mat_X, z1, z2, box, concat_label, wo_math, final_out)
             self.play(visual_group.animate.shift(UP*0.5))

        arrow = Arrow(box.get_bottom(), final_out.get_top(), color=WHITE, buff=0.1)
        self.play(GrowArrow(arrow), FadeIn(final_out))

        self.wait(2)
        
        if os.path.exists(filename):
            os.remove(filename)

%manim -qk -v warning MultiHeadAttentionScene

Manim Community v0.19.0